<a href="https://colab.research.google.com/github/jetsonmom/6.23_automobility_lesson/blob/main/%EC%A7%80%EC%97%B0%EC%A7%84%ED%95%99%EC%83%9D_%EB%A1%9C%EB%B3%B4%ED%94%8C%EB%A1%9C%EC%9A%B0_%EB%9D%BC%EB%B2%A8%EB%A7%81_%ED%95%9C_%ED%9B%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
base64 YOUR_IMAGE.jpg | curl -d @- \
  "https://detect.roboflow.com/0722_labeling-usrpl/1?api_key=JwvZQEBhBR5uPrwepqQW"

In [ ]:
!pip install roboflow
from roboflow import Roboflow

# Roboflow 모델 로드
rf = Roboflow(api_key="JwvZQEBhBR5uPrwepqQW")
project = rf.workspace().project("0722_labeling-usrpl")  # ← 이렇게!
model = project.version(1).model

In [ ]:
# YouTube 영상 다운로드 및 YOLOv11 객체 인식
# Google Colab에서 실행

# 1. 필요한 라이브러리 설치
!pip install yt-dlp roboflow ultralytics opencv-python

import cv2
import os
from roboflow import Roboflow
import numpy as np
from ultralytics import YOLO
import yt_dlp

# 2. YouTube 영상 다운로드 함수
def download_youtube_video(url, output_path="./"):
    """
    YouTube 영상을 다운로드하는 함수
    """
    ydl_opts = {
        'format': 'best[height<=720]',  # 720p 이하 품질로 다운로드
        'outtmpl': os.path.join(output_path, 'downloaded_video.%(ext)s'),
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            ydl.download([url])
            print("✅ 영상 다운로드 완료!")
            return True
        except Exception as e:
            print(f"❌ 다운로드 실패: {e}")
            return False

# 3. Roboflow에서 훈련된 모델 로드
def load_roboflow_model():
    """
    Roboflow에서 훈련된 모델을 로드
    """
    # 여러분의 API 키와 프로젝트 정보로 변경하세요
    rf = Roboflow(api_key="YOUR_API_KEY")  # 실제 API 키로 변경
    project = rf.workspace().project("0722_labeling-usrpl")  # 정확한 프로젝트 ID
    model = project.version(1).model
    return model

# 4. 영상에서 객체 인식 수행
def detect_objects_in_video(video_path, model, output_path="output_video.mp4"):
    """
    영상에서 객체를 인식하고 결과 저장
    """
    # 영상 읽기
    cap = cv2.VideoCapture(video_path)

    # 영상 정보 가져오기
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # 결과 영상 저장 설정
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"📹 총 {total_frames} 프레임 처리 시작...")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        # 매 5프레임마다 객체 인식 (성능 최적화)
        if frame_count % 5 == 0:
            try:
                # 임시 이미지 저장
                temp_img_path = "temp_frame.jpg"
                cv2.imwrite(temp_img_path, frame)

                # Roboflow 모델로 예측
                prediction = model.predict(temp_img_path, confidence=40, overlap=30)

                # 결과를 이미지에 그리기
                for box in prediction.json()['predictions']:
                    x1 = int(box['x'] - box['width']/2)
                    y1 = int(box['y'] - box['height']/2)
                    x2 = int(box['x'] + box['width']/2)
                    y2 = int(box['y'] + box['height']/2)

                    # 바운딩 박스 그리기
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

                    # 라벨과 신뢰도 표시
                    label = f"{box['class']}: {box['confidence']:.2f}"
                    cv2.putText(frame, label, (x1, y1-10),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                # 임시 파일 삭제
                os.remove(temp_img_path)

            except Exception as e:
                print(f"프레임 {frame_count} 처리 중 오류: {e}")

        # 결과 프레임 저장
        out.write(frame)

        # 진행상황 표시
        if frame_count % 100 == 0:
            progress = (frame_count / total_frames) * 100
            print(f"📊 진행률: {progress:.1f}% ({frame_count}/{total_frames})")

    # 리소스 해제
    cap.release()
    out.release()
    print(f"✅ 객체 인식 완료! 결과: {output_path}")

# 5. 메인 실행 함수
def main():
    # YouTube URL
    youtube_url = "https://www.youtube.com/watch?v=AxLmroTo3rQ"

    print("🎬 YouTube 영상 다운로드 시작...")
    if download_youtube_video(youtube_url):

        # 다운로드된 파일 찾기
        video_files = [f for f in os.listdir('.') if f.startswith('downloaded_video')]
        if video_files:
            video_path = video_files[0]
            print(f"📁 영상 파일: {video_path}")

            print("🤖 Roboflow 모델 로드 중...")
            model = load_roboflow_model()

            print("🔍 객체 인식 시작...")
            detect_objects_in_video(video_path, model, "result_with_detection.mp4")

            print("🎉 모든 작업 완료!")
            print("📺 결과 영상: result_with_detection.mp4")
        else:
            print("❌ 다운로드된 영상 파일을 찾을 수 없습니다.")

# 실행
if __name__ == "__main__":
    main()

# 추가: 결과 영상 재생 (Colab에서)
def play_result_video():
    """
    Colab에서 결과 영상 재생
    """
    from IPython.display import HTML
    from base64 import b64encode

    video_path = "result_with_detection.mp4"
    if os.path.exists(video_path):
        mp4 = open(video_path, 'rb').read()
        data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
        return HTML(f'<video width="800" controls><source src="{data_url}" type="video/mp4"></video>')
    else:
        print("결과 영상 파일이 없습니다.")

# 영상 재생
play_result_video()

In [ ]:
# YouTube 영상 다운로드 및 YOLOv11 객체 인식 - 완전판
# Google Colab에서 실행

# 1. 필요한 라이브러리 설치
!pip install yt-dlp roboflow ultralytics opencv-python

import cv2
import os
from roboflow import Roboflow
import numpy as np
from ultralytics import YOLO
import yt_dlp

# 2. YouTube 영상 다운로드 함수
def download_youtube_video(url, output_path="./"):
    """
    YouTube 영상을 다운로드하는 함수
    """
    ydl_opts = {
        'format': 'best[height<=720]',  # 720p 이하 품질로 다운로드
        'outtmpl': os.path.join(output_path, 'downloaded_video.%(ext)s'),
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            ydl.download([url])
            print("✅ 영상 다운로드 완료!")
            return True
        except Exception as e:
            print(f"❌ 다운로드 실패: {e}")
            return False

# 3. Roboflow에서 훈련된 모델 로드
def load_roboflow_model():
    """
    Roboflow에서 훈련된 모델을 로드
    """
    # 🚨 여기에 실제 API 키를 입력하세요! 🚨
    rf = Roboflow(api_key="JwvZQEBhBR5uPrwepqQW")  # ← 이곳 수정 필요!
    project = rf.workspace().project("0722_labeling-usrpl/1")  # 정확한 프로젝트 ID
    model = project.version(1).model
    return model

# 4. 영상에서 객체 인식 수행
def detect_objects_in_video(video_path, model, output_path="output_video.mp4"):
    """
    영상에서 객체를 인식하고 결과 저장
    """
    # 영상 읽기
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("❌ 영상 파일을 열 수 없습니다.")
        return False

    # 영상 정보 가져오기
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"📊 영상 정보: {width}x{height}, {fps}fps")

    # 결과 영상 저장 설정
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"📹 총 {total_frames} 프레임 처리 시작...")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        # 매 5프레임마다 객체 인식 (성능 최적화)
        if frame_count % 5 == 0:
            try:
                # 임시 이미지 저장
                temp_img_path = "temp_frame.jpg"
                cv2.imwrite(temp_img_path, frame)

                # Roboflow 모델로 예측
                prediction = model.predict(temp_img_path, confidence=40, overlap=30)

                # 결과를 이미지에 그리기
                predictions = prediction.json()['predictions']
                for box in predictions:
                    x1 = int(box['x'] - box['width']/2)
                    y1 = int(box['y'] - box['height']/2)
                    x2 = int(box['x'] + box['width']/2)
                    y2 = int(box['y'] + box['height']/2)

                    # 바운딩 박스 그리기
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

                    # 라벨과 신뢰도 표시
                    label = f"{box['class']}: {box['confidence']:.2f}"
                    cv2.putText(frame, label, (x1, y1-10),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                # 임시 파일 삭제
                if os.path.exists(temp_img_path):
                    os.remove(temp_img_path)

            except Exception as e:
                print(f"프레임 {frame_count} 처리 중 오류: {e}")

        # 결과 프레임 저장
        out.write(frame)

        # 진행상황 표시
        if frame_count % 100 == 0:
            progress = (frame_count / total_frames) * 100
            print(f"📊 진행률: {progress:.1f}% ({frame_count}/{total_frames})")

    # 리소스 해제
    cap.release()
    out.release()
    print(f"✅ 객체 인식 완료! 결과: {output_path}")
    return True

# 5. 메인 실행 함수
def main():
    """
    전체 과정을 실행하는 메인 함수
    """
    # YouTube URL
    youtube_url = "https://www.youtube.com/watch?v=AxLmroTo3rQ"

    print("🎬 YouTube 영상 다운로드 시작...")
    if download_youtube_video(youtube_url):

        # 다운로드된 파일 찾기
        video_files = [f for f in os.listdir('.') if f.startswith('downloaded_video')]
        if video_files:
            video_path = video_files[0]
            print(f"📁 영상 파일: {video_path}")

            print("🤖 Roboflow 모델 로드 중...")
            try:
                model = load_roboflow_model()
                print("✅ 모델 로드 완료!")

                print("🔍 객체 인식 시작...")
                if detect_objects_in_video(video_path, model, "result_with_detection.mp4"):
                    print("🎉 모든 작업 완료!")
                    print("📺 결과 영상: result_with_detection.mp4")
                    print("👀 play_result_video() 함수로 영상을 재생할 수 있습니다!")
                else:
                    print("❌ 객체 인식 중 오류가 발생했습니다.")

            except Exception as e:
                print(f"❌ 모델 로드 실패: {e}")
                print("🔑 API 키가 올바른지 확인해주세요!")
        else:
            print("❌ 다운로드된 영상 파일을 찾을 수 없습니다.")
    else:
        print("❌ YouTube 영상 다운로드에 실패했습니다.")

# 6. 결과 영상 재생 함수 (Colab용)
def play_result_video():
    """
    Colab에서 결과 영상 재생
    """
    from IPython.display import HTML
    from base64 import b64encode

    video_path = "result_with_detection.mp4"
    if os.path.exists(video_path):
        print("📺 결과 영상을 로드하는 중...")
        mp4 = open(video_path, 'rb').read()
        data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
        return HTML(f'<video width="800" controls><source src="{data_url}" type="video/mp4"></video>')
    else:
        print("❌ 결과 영상 파일이 없습니다.")
        print("먼저 main() 함수를 실행해주세요.")

# 7. 사용법 안내
def show_usage():
    """
    사용법 안내
    """
    print("=" * 50)
    print("🚀 YouTube YOLO 객체 인식 사용법")
    print("=" * 50)
    print("1️⃣ API 키 입력:")
    print('   load_roboflow_model() 함수의 api_key="여기" 부분 수정')
    print()
    print("2️⃣ 전체 실행:")
    print("   main()")
    print()
    print("3️⃣ 결과 재생:")
    print("   play_result_video()")
    print()
    print("🔑 Roboflow API 키는 다음에서 받으세요:")
    print("   https://app.roboflow.com → Settings → API")
    print("=" * 50)

# 8. 즉시 실행 (주석 해제하여 사용)
if __name__ == "__main__":
    # 사용법 먼저 표시
    show_usage()

    # API 키 입력 확인
    print("\n⚠️  실행하기 전에 API 키를 입력했는지 확인하세요!")
    print("준비되면 main() 함수를 실행하세요.")

    # 자동 실행하려면 아래 주석 해제
    main()

# 9. 개별 함수 테스트용
def test_download_only():
    """영상 다운로드만 테스트"""
    youtube_url = "https://www.youtube.com/watch?v=AxLmroTo3rQ"
    return download_youtube_video(youtube_url)

def test_model_only():
    """모델 로드만 테스트"""
    try:
        model = load_roboflow_model()
        print("✅ 모델 로드 성공!")
        return model
    except Exception as e:
        print(f"❌ 모델 로드 실패: {e}")
        return None

In [ ]:
# 차선 감지 모델을 위한 YouTube 영상 처리

import cv2
import os
from roboflow import Roboflow
import yt_dlp

def load_lane_detection_model():
    """
    차선 감지 모델 로드
    """
    rf = Roboflow(api_key="JwvZQEBhBR5uPrwepqQW")
    #project = rf.workspace().project("0722_labeling-usrpl/1")
    project = rf.workspace().project("0722_labeling-usrpl")
    model = project.version(1).model
    print("🛣️ 차선 감지 모델 로드 완료!")
    return model



def download_driving_video(url, output_path="./"):
    """
    드라이빙 영상 다운로드
    """
    ydl_opts = {
        'format': 'best[height<=720]',
        'outtmpl': os.path.join(output_path, 'driving_video.%(ext)s'),
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            ydl.download([url])
            print("✅ 드라이빙 영상 다운로드 완료!")
            return True
        except Exception as e:
            print(f"❌ 다운로드 실패: {e}")
            return False

def detect_lanes_in_video(video_path, model, output_path="lane_detection_result.mp4"):
    """
    영상에서 차선 감지
    """
    cap = cv2.VideoCapture(video_path)

    # 영상 정보
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"🎬 영상 정보: {width}x{height}, {fps}fps, {total_frames}프레임")

    # 출력 설정
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0
    lane_detections = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        # 매 3프레임마다 차선 감지 (더 자주)
        if frame_count % 3 == 0:
            try:
                temp_img_path = "temp_frame.jpg"
                cv2.imwrite(temp_img_path, frame)

                # 차선 감지 (낮은 신뢰도)
                prediction = model.predict(temp_img_path, confidence=30, overlap=50)
                predictions = prediction.json()['predictions']

                frame_lanes = len(predictions)
                lane_detections += frame_lanes

                if frame_lanes > 0:
                    print(f"🛣️ 프레임 {frame_count}: {frame_lanes}개 차선 감지")

                # 차선 그리기
                for lane in predictions:
                    x1 = int(lane['x'] - lane['width']/2)
                    y1 = int(lane['y'] - lane['height']/2)
                    x2 = int(lane['x'] + lane['width']/2)
                    y2 = int(lane['y'] + lane['height']/2)

                    # 차선은 보라색으로 표시
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 255), 3)

                    # 라벨
                    label = f"Lane: {lane['confidence']:.2f}"
                    cv2.rectangle(frame, (x1, y1-30), (x1+150, y1), (255, 0, 255), -1)
                    cv2.putText(frame, label, (x1+5, y1-8),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

                if os.path.exists(temp_img_path):
                    os.remove(temp_img_path)

            except Exception as e:
                print(f"❌ 프레임 {frame_count} 처리 오류: {e}")

        out.write(frame)

        # 진행상황
        if frame_count % 150 == 0:
            progress = (frame_count / total_frames) * 100
            print(f"📊 진행률: {progress:.1f}% (총 차선 감지: {lane_detections}개)")

    cap.release()
    out.release()

    print(f"✅ 완료! 총 {lane_detections}개 차선 감지")
    print(f"🎥 결과: {output_path}")

def main_lane_detection():
    """
    차선 감지 메인 함수
    """
    # 추천 YouTube URL들 (드라이빙 영상)
    driving_urls = [
        "https://www.youtube.com/watch?v=AxLmroTo3rQ",  # 기존 URL
        # 더 나은 드라이빙 영상이 있다면 교체하세요
    ]

    youtube_url = driving_urls[0]

    print("🚗 드라이빙 영상 다운로드 시작...")
    if download_driving_video(youtube_url):

        # 다운로드된 파일 찾기
        video_files = [f for f in os.listdir('.') if f.startswith('driving_video')]
        if video_files:
            video_path = video_files[0]
            print(f"📁 영상 파일: {video_path}")

            print("🛣️ 차선 감지 모델 로드 중...")
            try:
                model = load_lane_detection_model()

                print("🔍 차선 감지 시작...")
                detect_lanes_in_video(video_path, model, "lane_detection_result.mp4")

                print("🎉 차선 감지 완료!")
                print("📺 결과 영상: lane_detection_result.mp4")

            except Exception as e:
                print(f"❌ 모델 로드 실패: {e}")
                print("🔑 API 키를 확인해주세요!")
        else:
            print("❌ 다운로드된 영상 파일을 찾을 수 없습니다.")

def test_with_webcam_url():
    """
    Roboflow Visualize 페이지의 'Paste YouTube or Image URL' 기능 사용
    """
    print("🌐 웹 인터페이스 테스트:")
    print("1. Roboflow Visualize 페이지에서")
    print("2. 'Paste YouTube or Image URL' 입력창에")
    print("3. YouTube URL 붙여넣기")
    print("4. 차선이 잘 감지되는지 확인")

# 실행
if __name__ == "__main__":
    print("🛣️ 차선 감지 모드로 변경!")
    print("=" * 50)
    print("💡 이 모델은 차선(lane)을 감지하는 모델입니다.")
    print("📹 드라이빙 영상이나 도로 영상에서 가장 잘 작동합니다.")
    print()
    print("🚀 실행 방법:")
    print("1. API 키 입력")
    print("2. main_lane_detection() 실행")
    print("3. 또는 Roboflow 웹에서 test_with_webcam_url() 방법 시도")
    print("=" * 50)
    main_lane_detection()